# Local Invoice OCR on Google Colab
Select **Runtime > Change runtime type > T4 GPU**, reconnect, then run setup cells 1-5 in order. In cell 6 upload any invoice PDF to get its OCR/invoice JSON directly in Colab. No reference JSON, labels, Drive, or training are required. Accuracy mode reads original PDF pages with a stock local vision model and checks item arithmetic and OCR evidence; Fast mode uses spatial OCR only. Optional diagnostic cells follow the upload cell.

In [ ]:
#@title 1. Clone the OCR project
import os
import pathlib
import subprocess

PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ubaid-148/OCR.git', str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
os.chdir(PROJECT_DIR)
print('Project ready at', PROJECT_DIR)


In [ ]:
#@title 2. Install dependencies and verify the OCR device
import os, pathlib, shutil, subprocess, sys
REQUIRE_GPU_FOR_ACCURACY = True #@param {type:"boolean"}
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
print('GPU attached:', gpu_runtime, flush=True)
if REQUIRE_GPU_FOR_ACCURACY and not gpu_runtime:
    raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then run all cells again. Accuracy vision on CPU caused a 180-second timeout and must not silently fall back.')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-eng', 'tesseract-ocr-ara', 'tesseract-ocr-urd', 'ghostscript', 'unpaper', 'pngquant', 'zstd'], check=True)
# Keep OCR packages separate from Colab's preinstalled CUDA PyTorch.
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR / 'bin' / 'python')
# pip --python bootstraps pip even in a venv created without it.
ocr_pip = [sys.executable, '-m', 'pip', '--python', OCR_PYTHON]
subprocess.run([*ocr_pip, 'install', '-q', '--upgrade', 'pip'], check=True)
# ModelScope imports torch; its CPU build avoids a second CUDA/NCCL stack.
subprocess.run([*ocr_pip, 'install', '-q', 'torch==2.9.1+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
# CPU and GPU Paddle share a module: install exactly one distribution.
subprocess.run([*ocr_pip, 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=True)
requirements = [line.strip() for line in pathlib.Path('requirements.txt').read_text().splitlines() if line.strip() and not line.strip().startswith('paddlepaddle')]
subprocess.run([*ocr_pip, 'install', '-q', *requirements], check=True)
command = [*ocr_pip, 'install', '-q']
if gpu_runtime:
    command += ['paddlepaddle-gpu==3.3.1', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/']
else:
    command += ['paddlepaddle==3.3.1']
subprocess.run(command, check=True)
# Retry uncertain identifiers/numeric cells with bounded English OCR crops.
os.environ['OCR_TARGETED_RETRY'] = 'true'
os.environ['OCR_DEVICE'] = 'gpu:0' if gpu_runtime else 'cpu'
os.environ['VISION_REQUIRE_GPU'] = 'true'
os.environ.pop('OCR_PYTHON_EXE', None)
# Check in a fresh process so rerunning this cell cannot reuse an old Paddle import.
os.environ.setdefault('FLAGS_use_mkldnn', '0')
verification = subprocess.run(
    [OCR_PYTHON, '-u', str(PROJECT_DIR / 'check_ocr_runtime.py')],
    cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, errors='replace',
)
verification_log = pathlib.Path('/tmp/ocr-runtime-check.log')
verification_log.write_text(verification.stdout, encoding='utf-8')
print(verification.stdout, flush=True)
if verification.returncode:
    raise RuntimeError(
        f'OCR runtime verification failed (exit {verification.returncode}). '
        f'Full log: {verification_log}. Copy the error below:\n\n'
        + verification.stdout[-12000:]
    )
print('Dependencies ready. First upload loads OCR models; later uploads reuse them.')


In [ ]:
#@title 2b. Record the PaddleOCR runtime and model configuration
import json, os, subprocess

probe_code = ("import json,platform,paddle,paddleocr; "
              "print('OCR_RUNTIME_JSON='+json.dumps({'cuda':getattr(paddle,'cuda_version',lambda:None)(),"
              "'gpu_count':paddle.device.cuda.device_count() if paddle.is_compiled_with_cuda() else 0,"
              "'paddle':paddle.__version__,'paddleocr':paddleocr.__version__,'python':platform.python_version()}))")
probe = subprocess.run([OCR_PYTHON, "-c", probe_code], cwd=PROJECT_DIR,
                       capture_output=True, text=True, check=True)
runtime_line = next((line for line in probe.stdout.splitlines() if line.startswith("OCR_RUNTIME_JSON=")), None)
if runtime_line is None:
    raise RuntimeError(f"OCR runtime probe did not return JSON: {probe.stdout} {probe.stderr}")
runtime_info = json.loads(runtime_line.removeprefix("OCR_RUNTIME_JSON="))
gpu_count = runtime_info["gpu_count"]
runtime_device = os.environ.get("OCR_DEVICE", "auto")
if runtime_device == "auto":
    runtime_device = "gpu:0" if gpu_count else "cpu"

try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    ).stdout.strip()
except OSError:
    nvidia_smi = "unavailable"

OCR_MODEL_CONFIGURATION = {
    "GPU": nvidia_smi or ("available" if gpu_count else "not available"),
    "CUDA": runtime_info["cuda"] or "not exposed by Paddle",
    "Paddle": runtime_info["paddle"],
    "PaddleOCR": runtime_info["paddleocr"],
    "Python": runtime_info["python"],
    "Device": runtime_device,
    "Detection model": "PP-OCRv5_mobile_det",
    "English recognition model": "PP-OCRv5_mobile_rec",
    "Arabic recognition model": "arabic_PP-OCRv5_mobile_rec",
}
print(json.dumps(OCR_MODEL_CONFIGURATION, indent=2, ensure_ascii=False))
if runtime_device.startswith("gpu") and not gpu_count:
    print("WARNING: OCR_DEVICE requests GPU but Paddle reports no CUDA device.")


In [ ]:
#@title 4. Configure OCR-only mode
import os
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.environ['OCR_TARGETED_RETRY'] = 'false'
os.environ['OCR_FORCE_RASTER'] = 'true'
os.environ['OCR_DEVICE'] = 'gpu:0' if gpu_runtime else 'cpu'
os.environ['USE_LOCAL_AI'] = 'false'
os.environ['OLLAMA_URL'] = 'http://127.0.0.1:1/api/chat'
print('OCR-only mode enabled')
print('Paddle device:', os.environ['OCR_DEVICE'])
print('PDF text layer bypassed:', os.environ['OCR_FORCE_RASTER'])
print('Vision/parser pipeline disabled: raw PaddleOCR only')


In [ ]:
#@title 5. Optional: install and run Ollama with a local Qwen model
RUN_OLLAMA_SETUP = True #@param {type:"boolean"}
OLLAMA_MODEL = 'qwen2.5:14b-instruct' #@param {type:"string"}
import json
import os
import shutil
import subprocess
import time
import urllib.request

if RUN_OLLAMA_SETUP:
    if not shutil.which('ollama'):
        ollama_archive = '/tmp/ollama-linux-amd64.tar.zst'
        print('Installing Ollama...')
        urllib.request.urlretrieve('https://ollama.com/download/ollama-linux-amd64.tar.zst', ollama_archive)
        subprocess.run(['tar', '--zstd', '-xf', ollama_archive, '-C', '/usr'], check=True)
    if not shutil.which('ollama'):
        raise RuntimeError('Ollama installation failed: executable not found')

    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as response:
            ollama_ready = response.status == 200
    except OSError:
        ollama_ready = False
    if not ollama_ready:
        ollama_log = open('/tmp/ollama.log', 'a')
        ollama_process = subprocess.Popen(
            ['ollama', 'serve'], cwd='/content', start_new_session=True,
            stdout=ollama_log, stderr=subprocess.STDOUT,
        )
        for _ in range(60):
            try:
                with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as response:
                    if response.status == 200:
                        ollama_ready = True
                        break
            except OSError:
                time.sleep(1)
    if not ollama_ready:
        raise RuntimeError('Ollama did not start. Inspect /tmp/ollama.log')

    print(f'Pulling {OLLAMA_MODEL}; this may take several minutes...')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
    os.environ['USE_LOCAL_AI'] = 'true'
    os.environ['OLLAMA_URL'] = 'http://127.0.0.1:11434/api'

    placement = subprocess.run(['ollama', 'ps'], capture_output=True, text=True, check=True)
    print('Ollama model placement:')
    print(placement.stdout or '(model will load on first request)')
    if 'GPU' not in placement.stdout.upper():
        print('WARNING: Ollama does not currently report GPU placement. Check the Colab runtime and /tmp/ollama.log.')

    smoke_payload = json.dumps({
        'model': OLLAMA_MODEL,
        'prompt': 'Reply with exactly: OLLAMA_READY',
        'stream': False,
        'options': {'num_predict': 8},
    }).encode('utf-8')
    smoke_request = urllib.request.Request(
        'http://127.0.0.1:11434/api/generate', data=smoke_payload,
        headers={'Content-Type': 'application/json'}, method='POST',
    )
    with urllib.request.urlopen(smoke_request, timeout=180) as response:
        smoke_result = json.load(response)
    if smoke_result.get('error'):
        raise RuntimeError(f'Ollama smoke test failed: {smoke_result["error"]}')
    print('Ollama smoke test response:', smoke_result.get('response', '').strip())
    print(f'Ollama is ready with {OLLAMA_MODEL}.')
else:
    print('Ollama setup skipped. PaddleOCR-only cells remain available.')


In [ ]:
#@title 5. Upload one PDF and run raw PaddleOCR
import json
import os
import re
import subprocess
import tempfile
import time
from pathlib import Path
from google.colab import files

OCR_LANGUAGES = 'eng+ara' #@param ['eng+ara', 'eng', 'ara', 'eng+urd']
if 'OCR_PYTHON' not in globals():
    raise RuntimeError('Run cells 1-3 first so the OCR runtime is installed.')

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one PDF.')
original_name, pdf_bytes = next(iter(uploaded.items()))
if Path(original_name).suffix.lower() != '.pdf' or not pdf_bytes.startswith(b'%PDF-'):
    raise ValueError('The attached file must be a valid PDF.')

run_dir = Path(tempfile.mkdtemp(prefix='paddleocr-upload-'))
pdf_path = run_dir / Path(original_name).name
output_path = run_dir / 'raw_paddleocr.json'
pdf_path.write_bytes(pdf_bytes)

started = time.perf_counter()
process = subprocess.run(
    [OCR_PYTHON, '-u', str(PROJECT_DIR / 'coordinate_ocr.py'),
     str(pdf_path), str(output_path), OCR_LANGUAGES],
    cwd=PROJECT_DIR,
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)
elapsed = time.perf_counter() - started
if process.returncode:
    raise RuntimeError(process.stderr[-12000:] or process.stdout[-12000:] or 'PaddleOCR failed')

raw_ocr = json.loads(output_path.read_text(encoding='utf-8'))
raw_ocr['benchmark'] = {
    'input_filename': original_name,
    'elapsed_seconds': round(elapsed, 3),
    'device': raw_ocr.get('device', os.environ.get('OCR_DEVICE')),
    'engine': raw_ocr.get('engine'),
}
results_dir = PROJECT_DIR / 'benchmark_outputs' / 'uploads'
results_dir.mkdir(parents=True, exist_ok=True)
result_path = results_dir / (re.sub(r'[^A-Za-z0-9_.-]', '_', Path(original_name).stem) + '-raw-ocr.json')
result_path.write_text(json.dumps(raw_ocr, ensure_ascii=False, indent=2), encoding='utf-8')

evidence = []
for page in raw_ocr.get('pages', []):
    for word in page.get('words', []):
        evidence.append({
            'page': page.get('page'),
            'text': word.get('text'),
            'confidence': word.get('confidence'),
            'bbox': {
                'left': word.get('left'), 'top': word.get('top'),
                'width': word.get('width'), 'height': word.get('height'),
            },
        })
numeric_evidence = [item for item in evidence if re.search(r'\d', str(item.get('text', '')))]
print(f'File: {original_name}')
print(f'GPU/device: {raw_ocr.get("device", os.environ.get("OCR_DEVICE"))}')
print(f'OCR time: {elapsed:.3f} sec')
print(f'Pages: {len(raw_ocr.get("pages", []))}')
print(f'Detected text boxes: {len(evidence)}')
print(f'Numeric boxes: {len(numeric_evidence)}')
print('\nRAW OCR EVIDENCE (page, text, confidence, bbox):')
print(json.dumps(evidence, ensure_ascii=False, indent=2))
print('\nSaved JSON:', result_path)


In [ ]:
#@title 6. Save accurate, structured OCR JSON
from collections import defaultdict
from pathlib import Path

if 'raw_ocr' not in globals():
    raise RuntimeError('Run the upload/PaddleOCR cell first.')


def box_record(word, page_number):
    return {
        'page': page_number,
        'text': str(word.get('text', '')),
        'confidence': word.get('confidence'),
        'bbox': {
            'left': word.get('left'),
            'top': word.get('top'),
            'width': word.get('width'),
            'height': word.get('height'),
        },
    }


def row_key(box):
    height = float(box['bbox'].get('height') or 1)
    return round(float(box['bbox'].get('top') or 0) / max(height * 1.6, 12))

structured_pages = []
all_boxes = []
for page in raw_ocr.get('pages', []):
    page_number = page.get('page')
    boxes = [box_record(word, page_number) for word in page.get('words', [])]
    grouped = defaultdict(list)
    for box in boxes:
        grouped[row_key(box)].append(box)
    rows = []
    for group in grouped.values():
        cells = sorted(group, key=lambda item: float(item['bbox'].get('left') or 0))
        rows.append({
            'cells': cells,
            'text': ' '.join(cell['text'] for cell in cells),
            'min_confidence': min((cell['confidence'] for cell in cells if cell['confidence'] is not None), default=None),
        })
    rows.sort(key=lambda row: min(float(cell['bbox'].get('top') or 0) for cell in row['cells']))
    structured_pages.append({
        'page': page_number,
        'size': {'width': page.get('width'), 'height': page.get('height')},
        'render_dpi': page.get('render_dpi'),
        'rows': rows,
    })
    all_boxes.extend(boxes)

numeric_boxes = [box for box in all_boxes if any(character.isdigit() for character in box['text'])]
low_confidence = [box for box in all_boxes if box['confidence'] is not None and box['confidence'] < 80]
formatted_ocr = {
    'schema_version': 'paddleocr-evidence-1.0',
    'source': raw_ocr.get('benchmark', {}),
    'engine': raw_ocr.get('engine'),
    'language': raw_ocr.get('language'),
    'device': raw_ocr.get('device'),
    'timings_seconds': raw_ocr.get('timings_seconds'),
    'quality': {
        'status': 'needs_review' if low_confidence else 'ocr_evidence_ready',
        'pages': len(structured_pages),
        'text_boxes': len(all_boxes),
        'numeric_boxes': len(numeric_boxes),
        'low_confidence_boxes': len(low_confidence),
        'low_confidence_threshold': 80,
        'note': 'Rows are spatial OCR groupings. No invoice field or table value is guessed.',
    },
    'pages': structured_pages,
    'numeric_evidence': numeric_boxes,
    'low_confidence_evidence': low_confidence,
}
formatted_path = result_path.with_name(result_path.stem + '-formatted.json')
formatted_path.write_text(json.dumps(formatted_ocr, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(formatted_ocr, ensure_ascii=False, indent=2))
print('Saved formatted JSON:', formatted_path)
